# Vision · 범위 제한 기반 빈칸·부분 구현 문제

원본에 마스킹이 없어 **완성된 실습 코드를 바탕으로 새로 선정한 연습 문제**입니다. 실제 시험 문항이나 공식 배점을 의미하지 않습니다.

- ResNet: **0~5절**, 전체 셀15까지. 이번 요청의 “Chapter 6 이전”을 적용하여 6절 자체도 제외합니다.
- ViT: **6절 시작 전**, 전체 셀23까지. 원본 절3/4 순서가 뒤섞여 있어 셀 번호도 함께 기록합니다.
- DETR: 기존 확정 범위인 **13절 포함**, 전체 셀30까지.
- U-Net: 지정 노트북 전체. 설치·다운로드·긴 학습·그래프 꾸미기는 제외합니다.

근거: [시험 출제 포인트 PDF](../../AI인증시험_출제포인트.pdf) 1쪽 Vision의 레이어 이해, 예측/정답을 학습으로 연결, 데이터 구성, 이미지 변환 라이브러리, 평가 Metric. 문항별 셀 번호는 **Markdown 포함,1부터**입니다.

**사용법:** 준비 셀을 먼저 실행하고 각 문제에서 `MISSING` 또는 `raise NotImplementedError`를 올바른 코드로 교체 → 바로 아래 테스트 실행 → 실패하면 shape/축/순서 확인 → 마지막의 접힌 해설 확인. `# ????`는 원본에서 가져온 빈칸이 아니라 새 마스킹 표시입니다. 각 문제는 독립되어 이전 문제의 답이 필요 없습니다. 답을 수정하면 함수 셀을 먼저 다시 실행하세요. 예상 5~10분/문제(연습용 가안). 작성한 이 노트북을 제출하면 채점할 수 있습니다.

CPU·PyTorch·torchvision·einops·NumPy·Pillow만 사용합니다. 모델/데이터 다운로드 없음. 테스트는 shape뿐 아니라 수치, 학습 갱신, 잘못된 축, 다른 인자도 검사합니다. 통과는 작은 입력 검증이며 실전 정확도나 독립 숙달을 보장하지 않습니다.

**원본과 차이:** ViT FFN은 실제 코드의 ReLU 사용(주석의 GELU와 다름). U-Net DoubleConv는 실제 코드처럼 BN 없음. DETR attention reshape는 query/key 축이 뒤바뀐 설명·슬라이스를 교정하고 문제에서 명시합니다. ResNet AMP는 CPU에서 비활성 scaler로 순서를 확인합니다. 설치된 PyTorch 2.5.1/torchvision 0.20.1/einops 0.8.2에서 검증합니다.

이미지 텐서는 PyTorch의 실제 축 순서 **B,C,H,W**로 표기합니다. W,H를 구분하는 비정사각형 입력도 검사합니다. 이해가 안 되는 문제 번호·줄을 알려주면 작은 행렬 예시로 다시 설명할 수 있습니다.


In [ ]:
import copy
import math
import numpy as np
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

torch.set_num_threads(1)
MISSING = object()  # ???? 대신 실행 가능한 표식. 이 값을 올바른 코드로 바꾸세요.
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2023, 0.1994, 0.2010)

def close(actual, expected):
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)

class ModeProbe(nn.Module):
    # train/eval과 gradient 모드를 기록하는 작은 제공 분류기입니다.
    def __init__(self, width=4, classes=3):
        super().__init__()
        self.fc = nn.Linear(width, classes)
        self.seen = []
    def forward(self, x):
        self.seen.append((self.training, torch.is_grad_enabled()))
        return self.fc(x)


## R01. 학습·평가 전처리 구분

**PDF 점검:** 데이터 구성·이미지 변환 · **근거:** [01_ResNet18_CIFAR10.ipynb](../../vision/01_ResNet18_CIFAR10.ipynb) 전체 셀 6

size×size RandomCrop(padding=4), 좌우 반전, ToTensor, Normalize 순으로 train 변환을 구성하세요. test에는 ToTensor와 Normalize만 사용합니다. RGB PIL → float32 (3,size,size). 전달받은 mean/std를 사용하세요.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_r01(size, mean, std):
    train = T.Compose([
        MISSING,
        MISSING,
        MISSING,
        MISSING,
    ])
    test = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    return train, test

In [ ]:
def check_r01(fn):
    for size in (8, 12):
        mean, std = (0.2,0.3,0.4), (0.5,0.2,0.1)
        train, test = fn(size, mean, std)
        assert [type(t) for t in train.transforms] == [T.RandomCrop,T.RandomHorizontalFlip,T.ToTensor,T.Normalize]
        assert [type(t) for t in test.transforms] == [T.ToTensor,T.Normalize]
        assert train.transforms[0].size == (size,size) and train.transforms[0].padding == 4
        assert train.transforms[1].p == 0.5
        image = Image.fromarray(np.arange(size*size*3,dtype=np.uint8).reshape(size,size,3))
        expected=T.Normalize(mean,std)(T.ToTensor()(image))
        close(test(image),expected)
        torch.manual_seed(51)
        actual=train(image)
        torch.manual_seed(51)
        ref=T.Compose([T.RandomCrop(size,padding=4),T.RandomHorizontalFlip(),T.ToTensor(),T.Normalize(mean,std)])(image)
        close(actual,ref)
        assert actual.shape==(3,size,size) and actual.dtype==torch.float32

try:
    check_r01(solve_r01)
    print("PASS · R01")
except Exception as error:
    print("FAIL · R01 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

Compose는 변환을 순서대로 실행합니다. 예: uint8 픽셀255는 ToTensor 뒤1.0, mean=.5/std=.5이면 Normalize 뒤1.0입니다. 평가에는 무작위 증강을 넣지 않습니다. T의 원래 이름은 torchvision.transforms입니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

def solve_r01(size, mean, std):
    train = T.Compose([
        T.RandomCrop(size, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    test = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    return train, test
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## R02. DataLoader로 이미지·정답 묶기

**PDF 점검:** 데이터 구성 · **근거:** [01_ResNet18_CIFAR10.ipynb](../../vision/01_ResNet18_CIFAR10.ipynb) 전체 셀 6

제공된 train_set/test_set으로 로더를 만드세요. batch_size는 인자를 사용하고 train만 shuffle=True입니다. CPU 테스트를 위해 num_workers=0. 마지막 작은 배치는 버리지 않습니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_r02(train_set, test_set, batch_size):
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=MISSING, num_workers=0)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=MISSING, num_workers=0)
    return train_loader, test_loader

In [ ]:
def check_r02(fn):
    for bs in (2,3):
        ds=TensorDataset(torch.arange(5*3*4*4).float().reshape(5,3,4,4),torch.arange(5))
        tr,te=fn(ds,ds,bs)
        assert isinstance(tr.sampler,torch.utils.data.RandomSampler)
        assert isinstance(te.sampler,torch.utils.data.SequentialSampler)
        assert tr.num_workers==te.num_workers==0 and tr.batch_size==te.batch_size==bs
        batches=list(te)
        assert len(batches)==math.ceil(5/bs)
        close(torch.cat([y for x,y in batches]),torch.arange(5))
        assert batches[0][0].shape==(bs,3,4,4) and batches[0][1].dtype==torch.int64

try:
    check_r02(solve_r02)
    print("PASS · R02")
except Exception as error:
    print("FAIL · R02 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: 이미지5장, batch_size=2이면 배치 크기는2,2,1입니다. images=(B,3,H,W), labels=(B,) 정수 클래스 번호입니다. 원본의 worker/pin_memory는 실행환경 설정이라 여기서는 CPU용으로 고정했습니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

def solve_r02(train_set, test_set, batch_size):
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, test_loader
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## R03. 역정규화와 broadcasting

**PDF 점검:** 데이터 구성·이미지 변환 · **근거:** [01_ResNet18_CIFAR10.ipynb](../../vision/01_ResNet18_CIFAR10.ipynb) 전체 셀 7

정규화된 배치 x=(B,3,H,W)를 x*std+mean으로 되돌리고 표시용으로 [0,1]에 제한하세요. mean/std를 (1,3,1,1)로 만들어 채널별로 적용합니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_r03(x, mean_values, std_values):
    mean = torch.tensor(mean_values, device=x.device).view(MISSING)
    std = torch.tensor(std_values, device=x.device).view(MISSING)
    return MISSING

In [ ]:
def check_r03(fn):
    for b,h,w in [(1,4,5),(2,3,7)]:
        x=torch.linspace(-4,4,b*3*h*w).reshape(b,3,h,w)
        mean=(0.1,0.4,0.8);std=(0.2,0.3,0.5)
        expected=torch.stack([(x[:,c]*std[c]+mean[c]).clamp(0,1) for c in range(3)],dim=1)
        close(fn(x,mean,std),expected)

try:
    check_r03(solve_r03)
    print("PASS · R03")
except Exception as error:
    print("FAIL · R03 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: R채널 정규화 값2, std=.2, mean=.5 → .9. mean/std의1은 배치·높이·너비 방향으로 반복 적용됩니다. clamp는 정규화를 취소하는 연산이 아니라 표시 범위를 자르는 연산입니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

def solve_r03(x, mean_values, std_values):
    mean = torch.tensor(mean_values, device=x.device).view(1,3,1,1)
    std = torch.tensor(std_values, device=x.device).view(1,3,1,1)
    return (x * std + mean).clamp(0,1)
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## R04. CIFAR용 ResNet stem과 분류기

**PDF 점검:** 레이어 구성·동작 · **근거:** [01_ResNet18_CIFAR10.ipynb](../../vision/01_ResNet18_CIFAR10.ipynb) 전체 셀 9

제공된 ResNet18에서 conv1을 3→64, kernel=3,stride=1,padding=1,bias=False로 바꾸세요. maxpool을 Identity로 대체하고 FC 출력은 num_classes로 바꿉니다. 모델 생성·다운로드는 제공 코드가 처리합니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_r04(model, num_classes):
    model.conv1 = MISSING
    model.maxpool = MISSING
    model.fc = MISSING
    return model

In [ ]:
def check_r04(fn):
    for classes in (3,10):
        m=fn(torchvision.models.resnet18(weights=None),classes).eval()
        assert m.conv1.in_channels==3 and m.conv1.out_channels==64
        assert m.conv1.kernel_size==(3,3) and m.conv1.stride==(1,1) and m.conv1.padding==(1,1)
        assert m.conv1.bias is None and isinstance(m.maxpool,nn.Identity)
        assert m.fc.in_features==512 and m.fc.out_features==classes
        with torch.no_grad():
            x=torch.randn(2,3,32,32)
            assert m.conv1(x).shape==(2,64,32,32)
            assert m(x).shape==(2,classes)

try:
    check_r04(solve_r04)
    print("PASS · R04")
except Exception as error:
    print("FAIL · R04 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: (2,3,32,32)→stem(2,64,32,32)→ResNet stages→pool(2,512,1,1)→flatten(2,512)→FC(2,10)。Identity는 입력을 그대로 통과시킵니다. 내부 residual 구조는 torchvision이 제공합니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

def solve_r04(model, num_classes):
    model.conv1 = nn.Conv2d(3,64,kernel_size=3,stride=1,padding=1,bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features,num_classes)
    return model
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## R05. 분류 loss와 학습할 파라미터 연결

**PDF 점검:** 예측·정답을 학습으로 연결 · **근거:** [01_ResNet18_CIFAR10.ipynb](../../vision/01_ResNet18_CIFAR10.ipynb) 전체 셀 13

CrossEntropyLoss(label_smoothing 인자)와 SGD(lr,momentum,weight_decay 인자)를 생성하세요. optimizer는 model.parameters()를 갱신해야 합니다. logits=(B,C), labels=(B,) int64.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_r05(model, lr, momentum, weight_decay, smoothing):
    criterion = MISSING
    optimizer = MISSING
    return criterion, optimizer

In [ ]:
def check_r05(fn):
    for smoothing in (0.,0.1):
        model=nn.Linear(4,3)
        criterion,opt=fn(model,.03,.8,.002,smoothing)
        assert isinstance(criterion,nn.CrossEntropyLoss) and isinstance(opt,torch.optim.SGD)
        g=opt.param_groups[0]
        assert (g['lr'],g['momentum'],g['weight_decay'])==(.03,.8,.002)
        assert {id(p) for p in g['params']}=={id(p) for p in model.parameters()}
        logits=torch.tensor([[1.,-2.,.5],[.2,.3,2.]],requires_grad=True);labels=torch.tensor([0,2])
        close(criterion(logits,labels),F.cross_entropy(logits,labels,label_smoothing=smoothing))

try:
    check_r05(solve_r05)
    print("PASS · R05")
except Exception as error:
    print("FAIL · R05 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

CrossEntropyLoss에는 softmax 전 logits를 넣습니다. 예: labels=[2]는 세 번째 클래스가 정답이라는 뜻입니다. optimizer는 loss를 계산하지 않고 backward가 만든 gradient를 이용해 weight를 갱신합니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

def solve_r05(model, lr, momentum, weight_decay, smoothing):
    criterion = nn.CrossEntropyLoss(label_smoothing=smoothing)
    optimizer = torch.optim.SGD(model.parameters(),lr=lr,momentum=momentum,weight_decay=weight_decay)
    return criterion, optimizer
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## R06. AMP 학습 한 배치

**PDF 점검:** 예측·정답을 학습으로 연결 · **근거:** [01_ResNet18_CIFAR10.ipynb](../../vision/01_ResNet18_CIFAR10.ipynb) 전체 셀 11

train 모드→gradient 초기화→forward/loss→scaled backward→optimizer step→scaler update를 완성하세요. images=(B,4), labels=(B,)인 작은 분류기와 비활성 GradScaler를 제공합니다. GPU autocast 문맥은 CPU 최소 검증에서 생략합니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_r06(model, images, labels, optimizer, criterion, scaler):
    model.train()
    raise NotImplementedError("???? 이 줄을 완성하세요")
    logits = model(images)
    loss = MISSING
    raise NotImplementedError("???? 이 줄을 완성하세요")
    raise NotImplementedError("???? 이 줄을 완성하세요")
    raise NotImplementedError("???? 이 줄을 완성하세요")
    return loss.item()

In [ ]:
def check_r06(fn):
    for seed in (12,21):
        torch.manual_seed(seed)
        a=ModeProbe(); b=copy.deepcopy(a); a.eval()
        oa=torch.optim.SGD(a.parameters(),lr=.1); ob=torch.optim.SGD(b.parameters(),lr=.1)
        scaler=torch.amp.GradScaler('cuda',enabled=False)
        for size in (3,2):
            x=torch.randn(size,4);y=torch.arange(size)%3
            ob.zero_grad(set_to_none=True); expected=F.cross_entropy(b(x),y);expected.backward();ob.step()
            got=fn(a,x,y,oa,nn.CrossEntropyLoss(),scaler)
            assert abs(got-expected.item())<1e-6
            for p,q in zip(a.parameters(),b.parameters()):close(p,q)
        assert a.seen==[(True,True),(True,True)]

try:
    check_r06(solve_r06)
    print("PASS · R06")
except Exception as error:
    print("FAIL · R06 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: batch1의 gradient를 지우지 않고 batch2에서 backward하면 두 배치 gradient가 누적됩니다. zero_grad는 이를 초기화합니다. scaler.step이 optimizer를 호출해 weight를 갱신하고, scaler.update는 배율을 갱신합니다. CPU에서는 배율 처리를 비활성화하고 같은 순서를 검증합니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

def solve_r06(model, images, labels, optimizer, criterion, scaler):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(images)
    loss = criterion(logits,labels)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    return loss.item()
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## R07. 평가 모드·정확도·샘플 가중 평균

**PDF 점검:** 평가 Metric·학습/평가 구분 · **근거:** [01_ResNet18_CIFAR10.ipynb](../../vision/01_ResNet18_CIFAR10.ipynb) 전체 셀 11

빈칸을 채워 평가 loss와 top-1 accuracy를 반환하세요. 각 배치 criterion은 평균 loss입니다. 마지막 배치 크기가 달라도 전체 샘플 평균이 맞아야 합니다. eval과 no_grad 모두 필요합니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_r07(model, loader, criterion):
    raise NotImplementedError("???? 이 줄을 완성하세요")
    total_loss, correct, n = 0., 0, 0
    with MISSING:
        for images, labels in loader:
            logits=model(images)
            bs=images.size(0)
            total_loss += MISSING
            correct += MISSING
            n += bs
    return total_loss/n, correct/n

In [ ]:
def check_r07(fn):
    for bs in (2,3):
        torch.manual_seed(9);m=ModeProbe();m.train()
        x=torch.randn(5,4);y=torch.tensor([0,1,2,0,1])
        logits=F.linear(x,m.fc.weight,m.fc.bias).detach()
        expected_loss=F.cross_entropy(logits,y).item()
        expected_acc=(logits.argmax(1)==y).float().mean().item()
        got=fn(m,DataLoader(TensorDataset(x,y),batch_size=bs),nn.CrossEntropyLoss())
        assert abs(got[0]-expected_loss)<1e-6 and abs(got[1]-expected_acc)<1e-6
        assert all(state==(False,False) for state in m.seen)
        assert all(p.grad is None for p in m.parameters())

try:
    check_r07(solve_r07)
    print("PASS · R07")
except Exception as error:
    print("FAIL · R07 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: 2장 배치 loss1, 1장 배치 loss4 → 전체 평균=(1×2+4×1)/3=2입니다. (1+4)/2=2.5는 틀립니다. accuracy는 맞힌 샘플 수/전체 수이고 confidence 평균이 아닙니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

def solve_r07(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0., 0, 0
    with torch.no_grad():
        for images, labels in loader:
            logits=model(images)
            bs=images.size(0)
            total_loss += criterion(logits,labels).item() * bs
            correct += (logits.argmax(dim=1)==labels).sum().item()
            n += bs
    return total_loss/n, correct/n
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## R08. 추론 확률·클래스·confidence

**PDF 점검:** 레이어 활용·평가 Metric · **근거:** [01_ResNet18_CIFAR10.ipynb](../../vision/01_ResNet18_CIFAR10.ipynb) 전체 셀 15

logits=(B,C)를 클래스 축 softmax로 확률로 바꾸고, 각 이미지의 최대 확률 conf=(B,), 클래스 pred=(B,)를 반환하세요. 반환 순서는 pred,conf,probs입니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_r08(logits):
    probs = MISSING
    conf, pred = MISSING
    return pred,conf,probs

In [ ]:
def check_r08(fn):
    for x in [torch.tensor([[0.,0.,0.],[0.,math.log(2),0.]]),torch.tensor([[4.,1.],[1.,3.],[0.,-1.]])]:
        pred,conf,probs=fn(x)
        exp=x.exp()/x.exp().sum(1,keepdim=True)
        close(probs,exp);close(probs.sum(1),torch.ones(x.shape[0]))
        close(pred,x.argmax(1));close(conf,exp.gather(1,pred[:,None]).squeeze(1))
        assert pred.dtype==torch.int64

try:
    check_r08(solve_r08)
    print("PASS · R08")
except Exception as error:
    print("FAIL · R08 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: logits=[0,ln2,0]→확률=[.25,.5,.25]→pred=1,conf=.5. 정답 labels와 비교해야 accuracy를 알 수 있습니다. confidence=.5만으로 맞았다고 판단하지 않습니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader

def solve_r08(logits):
    probs = F.softmax(logits,dim=1)
    conf, pred = probs.max(dim=1)
    return pred,conf,probs
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>